In [46]:
import os
os.environ['HF_HOME'] = 'D:\\HuggingFace'
os.environ['TRANSFORMERS_CACHE'] = os.environ['HF_HOME']
os.environ['HUGGINGFACE_HUB_CACHE'] = os.environ['HF_HOME'] 

In [47]:
from warnings import filterwarnings
filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import pipeline
import re

sns.set_style('darkgrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.3 MB/s eta 0:00:00ta 0:00:01


# Обработка pdf

In [48]:
clean_data = pd.DataFrame({"Algo": [], "Analysis": [], "NN": [], "Optim": [], "SQL": []})

In [49]:
from PyPDF2 import PdfReader


def process_info(file):
    with open(file, "rb") as f:
        reader = PdfReader(f)
        text = ''.join([page.extract_text() for page in reader.pages])
    # Служебная информация
    text = re.sub(r'ISSN\s+\d{4}-\d{3,4}[^\n]*', '', text)
    text = re.sub(r'\d{4};\d{2}\(\d+\):\d+–\d+', '', text)

    # Авторы
    text = re.sub(r'[А-ЯЁ][а-яё]+\s+[А-ЯЁ][\.\s]+\s*[А-ЯЁ][\.\s]*', '', text)
    text = re.sub(r'[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ][\.\s]+\s*[А-ЯЁ][\.\s]*', '', text)
    text = re.sub(r'\d+[\s\w\.,–-]+(университет|институт|академия|центр)[^\n]*', '', text)

    # Сноски в квадратных скобках
    text = re.sub(r'\[\d+\]', '', text)  # [1], [2]
    text = re.sub(r'\[\d+[,-]\d+\]', '', text)  # [1-3], [4,5]
    text = re.sub(r'\[[A-Za-z]+\d*\]', '', text)  # [A1], [B]
    
    # email
    text = re.sub(r'\S+@\S+', '', text)
    
    # английские разделы 
    text = re.sub(r'Abstract[^\n]*[\s\S]*?(?=\n[А-ЯЁ]|$)', '', text)
    text = re.sub(r'Keywords[^\n]*[\s\S]*?(?=\n[А-ЯЁ]|$)', '', text)
    text = re.sub(r'For citation[^\n]*[\s\S]*?(?=\n[А-ЯЁ]|$)', '', text)
    
    # ссылки
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'DOI:\s*\S+', '', text)
    
    # библиография
    text = re.sub(r'Список\s+источников[\s\S]*', '', text)
    text = re.sub(r'References[\s\S]*', '', text)
    
    # спец.символы
    text = text.replace('\xa0', ' ').replace('•', '')
    text = re.sub(r'-\s+', '', text)  
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'•\s*\n', '', text)
    
    # оставшиеся английские фрагменты
    text = re.sub(r'(?:[A-Za-z-]+\s){3,}[A-Za-z-]*', '', text)
    
    return text

In [50]:
from pathlib import Path


def make_clean_pdfs(path):
    books = Path(path)
    files_to_process = []
    for book in books.iterdir():
        files_to_process.append(str(book))

    clean_files = []
    for file in files_to_process:
        clean_files.append(process_info(file))
        
    return clean_files

## Обработка всех классов и очистка pdf

In [51]:
algo_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Алгоритмы и структуры данных"
analysis_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Анализ данных"
nn_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/ИИ"
optim_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Методы оптимизации"
sql_path = "D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/SQL"

In [52]:
clean_data['Algo'] = make_clean_pdfs(algo_path)
print("____________DONE_____________")
clean_data['Analysis'] = make_clean_pdfs(analysis_path)
print("____________DONE_____________")
clean_data['NN'] = make_clean_pdfs(nn_path)
print("____________DONE_____________")
clean_data['Optim'] = make_clean_pdfs(optim_path)
print("____________DONE_____________")
clean_data['SQL'] = make_clean_pdfs(sql_path)
print("____________DONE_____________")

____________DONE_____________
____________DONE_____________
____________DONE_____________
____________DONE_____________
____________DONE_____________


In [53]:
clean_data.to_csv('clean_pdfs.csv', sep=',', index=False, encoding='utf-8-sig', escapechar='\\')

In [54]:
clean_data.shape

(5, 5)

# Создание обучающей выборки из эмбеддингов документов

In [55]:
text_dataset = pd.read_csv('D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/clean_pdfs.csv')

In [56]:
print(text_dataset.shape)
print(text_dataset.columns)

(5, 5)
Index(['Algo', 'Analysis', 'NN', 'Optim', 'SQL'], dtype='object')


In [80]:
len(text_dataset['SQL'][2])

1594644

In [42]:
text_dataset['Optim'][2]

'ОГЛАВЛЕНИЕ Предисловие ........... . Сnисок прюtятых обозначений . Введение . Глава 1. Задачи оптимизации 1.1. Основные понятия . . . . . . 1.2. Примеры задач оптимизации 1.3. Классы задач оnтимизации Вопросы для самопроверки Г л а в а 2. Методы одномерной минимизации 2.1. Предварительные замечания 2.2. Методы прямого поиска ... 2.3. Сравнение методов прямоrо поиска 2.4. Методы полиномиальной аппроксимации Вопросы для самопроверки . . . . . . . . Г л а в а 3. Многомерная безусловная минимизация 3.1. Методы спуска ........ . 3.2. Метод градиентного спуска .. 3.3. Минимизация квадратичной функции 3.4. Метод сопряженных направлений . 3.5. Метод Ньютона и его модификации 3.6. Квазиньютоновские методы . 3.7. Методы прямого поиска .. 3.8. Методы случайного поиска Вопросы для самопроверки 5 7 9 11 11 12 19 23 25 25 27 34 37 43 44 47 50 59 68 80 89 98 119 126 4 Оглавление Г л а в а 4. Аналитические методы нелинейноrо проrраммирования 128 4.1. Минимизация целевой функции на заданном множестве 

In [43]:
len(process_info('D://ПРОГА/Проектики/Github/Electronic_library/Диплом/Данные/Обучающая/Методы оптимизации/869636.pdf'))

133050

## Чанкование

In [32]:
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(separator='\n', chunk_size=500, chunk_overlap=10)
chunks = text_splitter.create_documents([text_dataset['Algo'][0]])

print(f'Всего чанков: {len(chunks)}')

Всего чанков: 1


In [36]:
text_dataset['Algo'][0]

'Лекции по алгоритмам и структурам данных. 11 февраля 2022 г.2Оглавление Лекция 1 13 1.1 Сложность алгоритма . . . . . . . . . . . . . . . . . . . . . . . . 14 1.1.1 Пример: поиск в массиве . . . . . . . . . . . . . . . . . . 16 1.1.2 Задача о наполнении рюкзака . . . . . . . . . . . . . . . 17 1.2 Исполнитель . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 19 1.2.1 Аппаратные исполнители . . . . . . . . . . . . . . . . . 20 1.2.2 Модулярная арифметика . . . . . . . . . . . . . . . . . . 21 1.3 Инварианты. Индуктивное программирование . . . . . . . . . 22 1.3.1 Индуктивные функции . . . . . . . . . . . . . . . . . . . 22 1.3.2 Доказательство корректности алгоритмов . . . . . . . . 23 1.4 Автоматы . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 24 1.5 Понятие интерфейс абстракции. . . . . . . . . . . . . . . . . . 26 1.5.1 Абстракция Последовательность . . . . . . . . . . . . . 26 1.5.2 Абстракция массив . . . . . . . . . . . . . . . . . . . . . 27 1.5.3 Интерфей

## Загрузка модели

## Получение эмбеддингов